# Comparação das máscaras de café por fonte de ground truth (estágio 03)

Quantifica a concordância entre as fontes habilitadas em `src/config.yaml` (MapBiomas, AlphaEarth) sobre o grid comum de 10 m: área de café por fonte, sobreposição, discordância e métricas de concordância (acordo global, IoU, F1, precisão, recall e kappa). Reutiliza as máscaras exportadas no estágio 02 (idempotência) e persiste o relatório JSON e a figura do diagnóstico em `MyDrive/tcc/artifacts/`, apoiando a escolha da fonte definitiva no estágio 04.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt` (incluindo rasterio, usado na leitura das máscaras), garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`.

In [ ]:
# Importa o pacote compartilhado, identifica a plataforma e resolve os caminhos de armazenamento.
from src import io
from src.config import get_config

platform = io.detect_platform()
storage_paths = io.resolve_storage_paths()
config = get_config()
print(f"Plataforma: {platform}")
print(f"Relatório do diagnóstico: {storage_paths['artifacts_metrics_ground_truth']}")
print(f"Figuras: {storage_paths['artifacts_figures']}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes e flags determinísticas do PyTorch de acordo com a configuração.
from src.utils import set_all_seeds, set_deterministic_flags

set_all_seeds(config["reproducibility"]["seed"])
set_deterministic_flags()
print(f"Seed fixada: {config['reproducibility']['seed']}")

## Fontes habilitadas e dependência das máscaras (estágio 02)

Lista as fontes habilitadas em `src/config.yaml` e o ano de referência, exige pelo menos duas fontes para o diagnóstico e verifica (sem reexportar) que o GeoTIFF binário de cada fonte está disponível no caminho canônico do estágio 02.

In [ ]:
# Lista as fontes habilitadas e verifica se as máscaras do estágio 02 existem.
from src.data.mask_utils import reference_year, source_mask_path

year = reference_year()
enabled_sources = [
    name for name, cfg in config["ground_truth"]["sources"].items() if cfg["enabled"]
]
print(f"Ano de referência: {year}")
print(f"Fontes habilitadas: {', '.join(enabled_sources) or 'nenhuma'}")

if len(enabled_sources) < 2:
    raise RuntimeError("O diagnóstico comparativo exige pelo menos duas fontes habilitadas.")

reference = enabled_sources[0]
other = next(source for source in enabled_sources if source != reference)

mask_paths = {name: source_mask_path(name, storage_paths) for name in enabled_sources}
for name, path in mask_paths.items():
    if not io.path_exists(path):
        raise FileNotFoundError(f"Máscara do estágio 02 não encontrada: {path}")
    print(f"Máscara {name}: {path}")

## Carregamento das máscaras no grid comum

Garante a cópia local de cada máscara (download no Kaggle apenas se ausente na sessão) e carrega os arrays binários no grid comum, definido pela primeira fonte — as demais são alinhadas por vizinho mais próximo quando necessário.

In [ ]:
# Baixa (quando necessário) e carrega as máscaras no grid comum da primeira fonte.
from src.data.mask_comparison import load_masks

local_mask_paths = {
    name: io.ensure_local_copy(path) for name, path in mask_paths.items()
}
mask_set = load_masks(local_mask_paths)
print(
    f"Grid comum: {mask_set.shape[0]}x{mask_set.shape[1]} px | "
    f"{mask_set.pixel_size_m:.0f} m | {mask_set.crs}"
)
for name in mask_set.masks:
    print(f"  {name}: {int(mask_set.masks[name].sum())} pixels de café")

## Diagnóstico comparativo

Calcula a área de café e a parcela do AOI por fonte, além das métricas de concordância entre a fonte de referência e as demais (acordo global, IoU, F1, precisão, recall e kappa).

In [ ]:
# Calcula o diagnóstico comparativo entre a fonte de referência e as demais.
from src.data.mask_comparison import compute_comparison

comparison = compute_comparison(mask_set, reference)
pair = comparison["pairs"][f"{reference}_vs_{other}"]
metrics = pair["metrics"]
print(f"Fonte de referência: {reference}")
for name in enabled_sources:
    print(
        f"  Área de café {name}: {comparison['area_km2'][name]:.2f} km² "
        f"({comparison['coffee_share'][name]:.2%} do AOI)"
    )
print(f"Acordo global: {metrics['overall_agreement']:.2%}")
print(f"IoU: {metrics['iou']:.2%} | F1: {metrics['f1']:.2%} | Kappa: {metrics['kappa']:.2f}")

## Relatório do diagnóstico

Persiste o relatório JSON do diagnóstico em `MyDrive/tcc/artifacts/metrics/ground_truth/`; execuções repetidas reutilizam o relatório já existente (idempotência).

In [ ]:
# Persiste o relatório JSON do diagnóstico (reutiliza se já existir).
from src.data.mask_comparison import save_report

report_path = save_report(comparison, storage_paths)

## Figura do diagnóstico

Renderiza e persiste a figura comparativa — fontes individuais e mapa de concordância de quatro classes — em `MyDrive/tcc/artifacts/figures/`; execuções repetidas reutilizam a figura já existente (idempotência).

In [ ]:
# Renderiza e persiste a figura do diagnóstico (reutiliza se já existir).
from src.data.mask_comparison import save_figure

figure_path = save_figure(mask_set, reference, storage_paths)

## Resumo da etapa

Exibe o resumo da comparação: ano de referência, fontes comparadas, grid comum, principais métricas de concordância e os caminhos dos artefatos persistidos.

In [ ]:
# Exibe o resumo da etapa de comparação das fontes.
summary = {
    "Ano de referência": year,
    "Fontes comparadas": ", ".join(enabled_sources),
    "Grid comum": f"{mask_set.shape[0]}x{mask_set.shape[1]} px ({mask_set.pixel_size_m:.0f} m)",
    "Acordo global": f"{metrics['overall_agreement']:.2%}",
    "IoU": f"{metrics['iou']:.2%}",
    "Relatório": str(report_path),
    "Figura": str(figure_path),
}
for key, value in summary.items():
    print(f"{key}: {value}")
print("Estágio 03 concluído.")